# 🤖 Automação com Python: navegador e desktop

### Mini curso prático · 2 horas · Windows e Mac

Computadores são ótimos em tarefas repetitivas — nós, nem tanto. Nesta aula vamos ensinar o
Python a **usar programas como uma pessoa usaria**: clicar, digitar, ler a tela e conferir resultados.

| Parte | Assunto | Tempo |
|---|---|---|
| 0 | Preparação | 10 min |
| A | **Selenium**: automatizar o navegador e escrever **testes automatizados** | 65 min |
| B | **Interface do sistema**: mouse, teclado e janelas do Windows e do Mac | 45 min |

![Visão geral: Selenium na Parte A; pyautogui, pywinauto e AppleScript na Parte B](diagramas/visao_geral.svg)

> **Antes da aula:** Python 3.10+, Jupyter e um navegador (Edge já vem no Windows; no Mac,
> Chrome). Detalhes no `README.md` desta pasta.

---
# Parte 0 — Preparação (10 min)

## 0.1 Instalando as bibliotecas

| Pacote | Para quê | Sistema |
|---|---|---|
| `selenium` | controlar o navegador | todos |
| `pytest` | rodar testes automatizados | todos |
| `pyautogui` + `pillow` | mouse, teclado e capturas de tela | todos |
| `pywinauto` | ler e controlar janelas do Windows "por dentro" | só Windows |

No Mac, o equivalente do `pywinauto` é o **AppleScript**, que já vem no sistema (comando `osascript`).

In [ ]:
import platform
import subprocess
import sys

SISTEMA = platform.system()  # "Windows", "Darwin" (= Mac) ou "Linux"
NO_WINDOWS = SISTEMA == "Windows"
NO_MAC = SISTEMA == "Darwin"

pacotes = ["selenium>=4.11", "pytest", "pyautogui", "pillow"]
if NO_WINDOWS:
    pacotes.append("pywinauto")

# Mesmo que "%pip install ...", mas aqui escolhemos os pacotes conforme o sistema.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pacotes], check=True)
print(f"✅ Instalado no {SISTEMA}: {', '.join(pacotes)}")

## 0.2 Pastas da aula

Usamos `pathlib` para montar caminhos: ele escreve `\` no Windows e `/` no Mac sozinho.

In [ ]:
import os
from pathlib import Path

AQUI = Path.cwd()
SITE = AQUI / "site_teste"
SAIDA = AQUI / "saida"          # screenshots e arquivos gerados pela aula
SAIDA.mkdir(exist_ok=True)

assert SITE.exists(), f"Abra o notebook de dentro da pasta aulas/automacao (estou em {AQUI})"
print("Pasta da aula:", AQUI)
print("Arquivos gerados vão para:", SAIDA)

---
# Parte A — Selenium: automatizando o navegador (65 min)

## A.1 O que é e para que serve

O **Selenium** controla um navegador de verdade (Chrome, Edge, Firefox, Safari) por meio de um
protocolo padrão chamado **WebDriver**. Tudo o que você faz com mouse e teclado numa página, ele faz
por código.

Usos mais comuns:
- ✅ **Testes automatizados**: garantir que o login, o carrinho, o cadastro... continuam
  funcionando a cada mudança no site — sem ninguém clicar em tudo à mão.
- 🔁 **Tarefas repetitivas**: preencher o mesmo formulário 200 vezes, baixar relatórios todo dia.
- 📥 **Coletar dados** de páginas que só funcionam com JavaScript.

![Como o Selenium controla o navegador](diagramas/selenium_arquitetura.svg)

> ⚖️ **Use com responsabilidade.** Automatize sites que são **seus**, que **permitem** isso nos
> termos de uso, ou páginas feitas para praticar (como a nossa). Não sobrecarregue servidores alheios.

## A.2 O nosso site de treino

A pasta `site_teste/` tem uma "loja" feita para esta aula: login, formulário, tabela e um botão
que demora para responder. Vamos servi-la localmente, sem depender da internet.

> 💡 O servidor roda numa *thread* (em paralelo), então o notebook continua livre para usar.

In [ ]:
import functools
import http.server
import threading


class ServidorSilencioso(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args):  # não imprime uma linha por requisição
        pass


handler = functools.partial(ServidorSilencioso, directory=str(SITE))
servidor = http.server.ThreadingHTTPServer(("127.0.0.1", 0), handler)  # porta 0 = o sistema escolhe uma livre
threading.Thread(target=servidor.serve_forever, daemon=True).start()

URL = f"http://127.0.0.1:{servidor.server_port}/index.html"
print("🛒 Loja Teste no ar em", URL)

## A.3 Abrindo o navegador

Desde a versão 4.11, o Selenium tem o **Selenium Manager**: ele descobre qual navegador você tem e
**baixa o driver certo sozinho**. Antigamente era preciso baixar o `chromedriver` à mão e acertar a
versão — se você achar tutoriais com isso, estão desatualizados.

Escolha o navegador na variável `NAVEGADOR`:

| Valor | Observação |
|---|---|
| `"edge"` | já vem instalado no **Windows** — padrão lá |
| `"chrome"` | padrão no **Mac** |
| `"firefox"` | funciona nos dois, se estiver instalado |
| `"safari"` | só Mac; antes rode uma vez no Terminal `safaridriver --enable` (pede a senha) |

`HEADLESS = True` roda o navegador **sem janela** (é como os testes rodam em servidores). Na aula,
deixe `False` para ver a mágica acontecer.

In [ ]:
from selenium import webdriver

NAVEGADOR = "edge" if NO_WINDOWS else "chrome"
HEADLESS = os.getenv("HEADLESS") == "1"  # False, a não ser que você defina HEADLESS=1


def abrir_navegador(nome: str = NAVEGADOR, headless: bool = HEADLESS):
    nome = nome.lower()
    if nome == "chrome":
        opcoes = webdriver.ChromeOptions()
        if headless:
            opcoes.add_argument("--headless=new")
        return webdriver.Chrome(options=opcoes)
    if nome == "edge":
        opcoes = webdriver.EdgeOptions()
        if headless:
            opcoes.add_argument("--headless=new")
        return webdriver.Edge(options=opcoes)
    if nome == "firefox":
        opcoes = webdriver.FirefoxOptions()
        if headless:
            opcoes.add_argument("-headless")
        return webdriver.Firefox(options=opcoes)
    if nome == "safari":
        return webdriver.Safari()
    raise ValueError(f"Navegador desconhecido: {nome}")


driver = abrir_navegador()
driver.get(URL)
print("Título da página:", driver.title)

Uma janela do navegador abriu com uma faixa avisando que ele está **sendo controlado por software
automatizado**. Não feche essa janela à mão: o `driver` é o nosso "controle remoto" dela.

Uma captura de tela, para registro:

In [ ]:
arquivo = SAIDA / "01_pagina_inicial.png"
driver.save_screenshot(str(arquivo))
print("📸 salvo em", arquivo)

## A.4 Encontrando elementos: os localizadores

Para clicar ou digitar, primeiro precisamos **achar** o elemento na página. O Selenium oferece
várias estratégias (`By`):

| `By.` | Procura por | Exemplo no nosso HTML |
|---|---|---|
| `ID` | atributo `id` (único na página) | `<input id="usuario">` |
| `NAME` | atributo `name` | `<input name="senha">` |
| `CSS_SELECTOR` | seletor CSS, como no estilo da página | `[data-testid=entrar]`, `#painel .ok` |
| `XPATH` | caminho no documento; o mais poderoso e o mais frágil | `//button[text()='Entrar']` |
| `TAG_NAME`, `CLASS_NAME`, `LINK_TEXT` | tag, classe, texto do link | |

![Os localizadores apontando para os elementos do formulário de login](diagramas/localizadores.svg)

> 💡 **Qual usar?** Prefira `ID` ou um atributo feito para testes (`data-testid`). Evite XPaths
> longos como `/html/body/div[2]/section/form/button`: qualquer mudança no layout quebra tudo.

Para descobrir o `id` de algo: no navegador, clique com o botão direito no elemento → **Inspecionar**.

In [ ]:
from selenium.webdriver.common.by import By

campo_usuario = driver.find_element(By.ID, "usuario")
campo_senha = driver.find_element(By.NAME, "senha")
botao_css = driver.find_element(By.CSS_SELECTOR, "[data-testid=entrar]")
botao_xpath = driver.find_element(By.XPATH, "//button[text()='Entrar']")

print("Mesmo botão pelos dois caminhos?", botao_css == botao_xpath)
print("Tag do campo usuário:", campo_usuario.tag_name, "| tipo:", campo_usuario.get_attribute("type"))

## A.5 Interagindo: digitar e clicar

| Método | Faz |
|---|---|
| `.send_keys("texto")` | digita no elemento |
| `.clear()` | apaga o conteúdo |
| `.click()` | clica |
| `.text` | o texto visível |
| `.get_attribute("value")` | o valor de um campo |
| `.is_displayed()` | está visível? |

Primeiro, com a senha **errada**:

In [ ]:
campo_usuario.send_keys("aluno")
campo_senha.send_keys("senha-errada")
botao_css.click()

erro = driver.find_element(By.ID, "erro-login")
print("Mensagem de erro visível?", erro.is_displayed(), "→", erro.text)

In [ ]:
campo_senha.clear()
campo_senha.send_keys("selenium123")
botao_css.click()

print(driver.find_element(By.ID, "boas-vindas").text)
print("Novo título:", driver.title)

## A.6 Formulários: listas, botões de opção, caixas de seleção

- `<select>` (lista suspensa) tem um ajudante próprio: `Select`.
- Botões de opção (*radio*) e caixas (*checkbox*) são só `.click()`.

In [ ]:
from selenium.webdriver.support.ui import Select

driver.find_element(By.ID, "cliente").send_keys("Turma de automação")
Select(driver.find_element(By.ID, "produto")).select_by_visible_text("Queijo minas")

quantidade = driver.find_element(By.ID, "quantidade")
quantidade.clear()
quantidade.send_keys("4")

driver.find_element(By.CSS_SELECTOR, "input[name=entrega][value=casa]").click()
driver.find_element(By.ID, "presente").click()
driver.find_element(By.CSS_SELECTOR, "[data-testid=enviar-pedido]").click()

print(driver.find_element(By.ID, "mensagem-pedido").text)
driver.save_screenshot(str(SAIDA / "02_pedido.png"));

## A.7 Lendo dados da página

`find_elements` (no plural) devolve uma **lista** — vazia se não achar nada, em vez de dar erro.

In [ ]:
cabecalho = [th.text for th in driver.find_elements(By.CSS_SELECTOR, "#tabela-pedidos thead th")]
pedidos = []
for linha in driver.find_elements(By.CSS_SELECTOR, "#tabela-pedidos tbody tr"):
    celulas = [td.text for td in linha.find_elements(By.TAG_NAME, "td")]
    pedidos.append(dict(zip(cabecalho, celulas)))

for p in pedidos:
    print(p)

## A.8 Esperas: o erro nº 1 de quem começa

Páginas reais carregam coisas **depois** de um tempo (respostas de servidor, animações). O botão
**Carregar relatório** do nosso site demora entre 1,5 e 3 segundos — de propósito. Se procurarmos o
resultado imediatamente:

In [ ]:
from selenium.common.exceptions import NoSuchElementException

driver.find_element(By.ID, "carregar-relatorio").click()
try:
    driver.find_element(By.ID, "relatorio")
except NoSuchElementException:
    print("💥 NoSuchElementException: o relatório ainda não existia quando procuramos!")

A tentação é pôr um `time.sleep(2)`. **Não faça isso:**
- se o site for mais rápido, você perde tempo à toa (multiplique por 500 testes...);
- se for mais lento (rede ruim, computador ocupado), o teste quebra **às vezes** — o pior tipo de
  erro, o teste "instável" (*flaky*).

O certo é uma **espera explícita**: "espere **até** esta condição ser verdadeira, por no máximo N
segundos". O `WebDriverWait` verifica de tempos em tempos (a cada 0,5 s, por padrão) e segue em
frente assim que der.

![time.sleep × WebDriverWait numa linha do tempo](diagramas/esperas.svg)

In [ ]:
import time

from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

espera = WebDriverWait(driver, timeout=10)

inicio = time.time()
driver.find_element(By.ID, "carregar-relatorio").click()
relatorio = espera.until(EC.visibility_of_element_located((By.ID, "relatorio")))
print(f"{relatorio.text}  (esperou {time.time() - inicio:.1f} s — nem mais, nem menos)")

Condições mais usadas de `expected_conditions` (`EC`):

| Condição | Espera até... |
|---|---|
| `visibility_of_element_located(loc)` | o elemento existir **e** estar visível |
| `element_to_be_clickable(loc)` | dar para clicar |
| `text_to_be_present_in_element(loc, "texto")` | aparecer um texto |
| `title_contains("texto")` | o título da aba mudar |
| `invisibility_of_element_located(loc)` | sumir (ex.: um "carregando...") |

## 🎯 Exercício A1

1. Recarregue a página (`driver.get(URL)`), faça login e crie **três pedidos** a partir da lista abaixo,
   usando um `for`.
2. Clique em **Carregar relatório** e confira com `assert` que o total é 6.

<details>
<summary>👀 Ver solução</summary>

```python
driver.get(URL)
driver.find_element(By.ID, "usuario").send_keys("aluno")
driver.find_element(By.ID, "senha").send_keys("selenium123")
driver.find_element(By.CSS_SELECTOR, "[data-testid=entrar]").click()

for cliente, produto in novos:
    driver.find_element(By.ID, "cliente").send_keys(cliente)
    Select(driver.find_element(By.ID, "produto")).select_by_visible_text(produto)
    driver.find_element(By.CSS_SELECTOR, "[data-testid=enviar-pedido]").click()

driver.find_element(By.ID, "carregar-relatorio").click()
texto = espera.until(EC.visibility_of_element_located((By.ID, "relatorio"))).text
assert texto == "Total de pedidos: 6", texto
print("✅", texto)
```
</details>

In [ ]:
novos = [("Bia", "Café especial"), ("Caio", "Doce de leite"), ("Duda", "Queijo minas")]

# Sua resposta aqui

## A.9 De script a teste automatizado

O que fizemos até aqui é um **script**: roda e mostra coisas na tela; alguém precisa olhar e julgar.
Um **teste automatizado** decide sozinho se passou ou falhou, com `assert`:

In [ ]:
def teste_login_errado():
    driver.get(URL)
    driver.find_element(By.ID, "usuario").send_keys("aluno")
    driver.find_element(By.ID, "senha").send_keys("errada")
    driver.find_element(By.CSS_SELECTOR, "[data-testid=entrar]").click()
    erro = espera.until(EC.visibility_of_element_located((By.ID, "erro-login")))
    assert erro.text == "Usuário ou senha inválidos."


def teste_login_certo():
    driver.get(URL)
    driver.find_element(By.ID, "usuario").send_keys("aluno")
    driver.find_element(By.ID, "senha").send_keys("selenium123")
    driver.find_element(By.CSS_SELECTOR, "[data-testid=entrar]").click()
    assert espera.until(EC.title_contains("Painel"))


for teste in [teste_login_errado, teste_login_certo]:
    try:
        teste()
        print(f"✅ {teste.__name__}")
    except AssertionError as e:
        print(f"❌ {teste.__name__}: {e}")

Funciona, mas estamos reinventando a roda. O **pytest** é a ferramenta padrão de testes em Python e
já cuida de tudo isso: descobre os testes (funções que começam com `test_`), roda, mostra o relatório
e ainda oferece **fixtures** — preparação reaproveitável, como "abrir o navegador" e "subir o site".

### Os arquivos

```
testes/
├── conftest.py    ← fixtures: sobe o site e abre o navegador (uma vez para todos os testes)
└── test_site.py   ← os testes + um Page Object
```

### Page Object: separar o "como" do "o quê"

Repare como os testes acima repetem `find_element(By.ID, "usuario")...`. Se o `id` mudar, temos que
caçar em todo lugar. O padrão **Page Object** junta numa classe *como* interagir com a página; os
testes só dizem *o que* querem:

![Testes, Page Object e página](diagramas/page_object.svg)

```python
def test_login_correto_mostra_boas_vindas(loja):
    loja.login("aluno", "selenium123")
    assert loja.boas_vindas() == "Bem-vindo(a), aluno!"
```

In [ ]:
print((AQUI / "testes" / "test_site.py").read_text(encoding="utf-8"))

### Rodando o pytest

No terminal, de dentro de `aulas/automacao`, o comando é `python -m pytest testes -v`.
Daqui do notebook usamos o `subprocess`. Os testes rodam *headless* (sem janela), por isso são rápidos.

In [ ]:
resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "testes", "-v", "--color=no", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    env={**os.environ, "HEADLESS": "1", "NAVEGADOR": NAVEGADOR},
)
print(resultado.stdout[-2500:] or resultado.stderr[-2500:])

## 🎯 Exercício A2

1. Em `testes/test_site.py`, escreva `test_pedido_para_presente`: crie um pedido com
   `presente=True` e confira que a mensagem termina com `(para presente).`
2. **Quebre o site de propósito**: em `site_teste/index.html`, troque `selenium123` por outra senha.
   Rode o pytest. Quais testes falham? A mensagem de erro ajuda a achar o problema? (Desfaça depois!)

<details>
<summary>👀 Ver solução do item 1</summary>

```python
def test_pedido_para_presente(loja):
    loja.login("aluno", "selenium123")
    mensagem = loja.criar_pedido("Duda", "Café especial", presente=True)
    assert mensagem.endswith("(para presente).")
```
</details>

## 🎯 Exercício A3 — um site de verdade

O próprio projeto Selenium mantém uma página para treinar:
**https://www.selenium.dev/selenium/web/web-form.html**

Preencha o campo de texto, a senha, a área de texto e o *dropdown*, clique em **Submit** e
imprima a mensagem que aparece.

<details>
<summary>👀 Ver solução</summary>

```python
driver.get("https://www.selenium.dev/selenium/web/web-form.html")
driver.find_element(By.ID, "my-text-id").send_keys("Olá, Selenium")
driver.find_element(By.NAME, "my-password").send_keys("123456")
driver.find_element(By.NAME, "my-textarea").send_keys("Automatizado com Python")
Select(driver.find_element(By.NAME, "my-select")).select_by_value("2")
driver.find_element(By.CSS_SELECTOR, "button[type=submit]").click()
print(espera.until(EC.visibility_of_element_located((By.ID, "message"))).text)
```
</details>

In [ ]:
# Sua resposta aqui

Terminamos com o navegador. **Sempre** feche o driver no fim: `quit()` encerra a janela e o
processo do driver (`close()` fecharia só a aba).

In [ ]:
driver.quit()
servidor.shutdown()
print("👋 navegador fechado e servidor desligado")

---
# Parte B — Automatizando a interface do sistema (45 min)

O Selenium só enxerga **dentro** do navegador. E o Bloco de Notas, a Calculadora, aquele sistema
antigo da empresa que só roda no desktop? Existem duas abordagens:

| | **"Olhos e mãos"** | **"Por dentro"** (árvore de acessibilidade) |
|---|---|---|
| Como funciona | move o mouse para coordenadas, aperta teclas, olha pixels | pergunta ao sistema quais janelas, botões e campos existem |
| Ferramenta | `pyautogui` (Windows **e** Mac) | `pywinauto` (Windows) · AppleScript (Mac) |
| Vantagem | funciona com **qualquer** coisa na tela | **robusto**: acha o botão "7" esteja onde estiver |
| Problema | frágil: janela mudou de lugar, resolução diferente, tudo quebra | depende do app expor essas informações |

![pyautogui clica em coordenadas; a árvore de acessibilidade acha o botão pelo auto_id](diagramas/duas_abordagens.svg)

> A "árvore de acessibilidade" existe para os **leitores de tela** usados por pessoas cegas.
> Programas de automação aproveitam a mesma informação.

## ⚠️ Permissões no Mac (faça antes!)

O macOS protege o controle do computador. Na primeira vez, ele vai pedir — ou, pior, vai falhar
em silêncio. Libere o programa em que o notebook roda (**Terminal**, **VS Code**, **iTerm**...) em
**Ajustes do Sistema → Privacidade e Segurança**:
- **Acessibilidade** — para mover o mouse, digitar e ler a árvore de elementos;
- **Gravação de Tela** — para as capturas de tela (sem ela, a imagem sai só com o papel de parede);
- **Automação** — o macOS pergunta sozinho na primeira vez que um script controla um app; diga **OK**.

Depois de liberar, **feche e abra de novo** o programa.

No Windows não há nada a liberar.

## B.1 `pyautogui`: olhos e mãos

### 🛑 Antes de tudo: o freio de emergência

Um script que controla o mouse pode sair do controle. Com `FAILSAFE = True` (o padrão),
**jogue o mouse num dos cantos da tela** e o `pyautogui` para na hora com um erro.
`PAUSE` põe uma pausa entre cada ação, para dar tempo de ver (e de reagir).

In [ ]:
import pyautogui

pyautogui.FAILSAFE = True  # mouse num canto da tela = parar tudo
pyautogui.PAUSE = 0.3      # segundos entre uma ação e outra

# As teclas de atalho mudam: Ctrl no Windows, Command (⌘) no Mac.
TECLA_ATALHO = "command" if NO_MAC else "ctrl"

largura, altura = pyautogui.size()
print(f"Tela: {largura} × {altura} pontos")
print("O mouse está em:", pyautogui.position())

> 🍎 Em telas Retina, o `size()` devolve **pontos**, não pixels: uma tela de 2880 × 1800 pixels
> aparece como 1440 × 900. As capturas de tela, porém, vêm em pixels. Lembre disso se for
> combinar as duas coisas.

### Mouse

Rode e **solte o mouse**: ele vai desenhar um quadrado no meio da tela.

In [ ]:
cx, cy = largura // 2, altura // 2
lado = 200

pyautogui.moveTo(cx - lado // 2, cy - lado // 2, duration=0.5)
for dx, dy in [(lado, 0), (0, lado), (-lado, 0), (0, -lado)]:
    pyautogui.moveRel(dx, dy, duration=0.4)
print("🟥 quadrado feito! O mouse está em", pyautogui.position())

Outras ações de mouse: `click(x, y)`, `doubleClick()`, `rightClick()`, `dragTo(x, y)`, `scroll(-5)`.

### Captura de tela

In [ ]:
try:
    captura = pyautogui.screenshot()
    arquivo = SAIDA / "03_minha_tela.png"
    captura.save(arquivo)
    print(f"📸 {captura.size[0]} × {captura.size[1]} pixels salvos em {arquivo}")
    # No Mac: se a imagem só tiver o papel de parede, falta a permissão de Gravação de Tela.
except Exception as erro:  # no Mac, sem a permissão, o comando de captura falha
    print(f"🔒 Não consegui capturar a tela ({type(erro).__name__}).")
    print("   No Mac: Ajustes do Sistema → Privacidade e Segurança → Gravação de Tela → ative o")
    print("   programa que roda este notebook e reinicie-o.")

### Teclado: escrevendo num editor de texto

Vamos abrir o editor de texto do sistema (**Bloco de Notas** no Windows, **TextEdit** no Mac) com um
arquivo novo e digitar nele.

Para abrir um programa usamos o `subprocess`, cada sistema com o seu comando:

In [ ]:
def abrir_no_editor(arquivo: Path):
    """Abre o arquivo no editor de texto padrão do sistema."""
    arquivo.touch()  # cria o arquivo vazio, se não existir
    if NO_WINDOWS:
        subprocess.Popen(["notepad.exe", str(arquivo)])
    elif NO_MAC:
        subprocess.run(["open", "-a", "TextEdit", str(arquivo)], check=True)
    else:
        subprocess.Popen(["xdg-open", str(arquivo)])
    time.sleep(2)  # dar tempo da janela aparecer (veremos jeitos melhores na B.2)


nota_pyautogui = SAIDA / "nota_pyautogui.txt"
print("Editor do sistema:", "Bloco de Notas" if NO_WINDOWS else "TextEdit" if NO_MAC else "padrão")

Agora o ponto **mais delicado** do `pyautogui`: ele digita na janela que estiver **em foco**, seja
ela qual for. Por isso, depois de rodar a célula abaixo, **não clique em nada** até ela terminar.

> ⚠️ O `pyautogui.write()` só digita bem caracteres **sem acento** (ele simula teclas físicas de
> um teclado americano). Para texto em português, o truque é **copiar para a área de transferência
> e colar** com o atalho.

In [ ]:
import pyperclip  # vem junto com o pyautogui

abrir_no_editor(nota_pyautogui)

pyautogui.write("Texto digitado pelo pyautogui, tecla por tecla.\n", interval=0.03)

pyperclip.copy("Já este texto, com acentuação, foi colado: ação, coração, pão de queijo.\n")
pyautogui.hotkey(TECLA_ATALHO, "v")

pyautogui.hotkey(TECLA_ATALHO, "s")  # salvar
time.sleep(1)
print("✍️  Pronto! Confira a janela do editor.")

Funcionou? Ótimo. Agora pense em tudo o que poderia dar errado: uma notificação roubou o foco;
o editor demorou mais de 2 segundos para abrir; o teclado está em outro idioma... É por isso que a
próxima abordagem existe.

Feche a janela do editor antes de continuar.

> 💡 O `pyautogui` também acha imagens na tela (`pyautogui.locateOnScreen("botao.png")`) para
> clicar nelas. É o último recurso para programas que não expõem nada por dentro.

---
## B.2 Por dentro: a árvore de acessibilidade

Aqui os caminhos se separam por sistema — mas a **ideia é a mesma**:
1. encontrar a janela do programa **pelo nome**, não pela posição;
2. encontrar os elementos dentro dela (campos, botões) **pelo tipo e pelo identificador**;
3. agir neles diretamente.

Leia a seção do seu sistema; a do outro serve de comparação.

### 🪟 Windows: `pywinauto`

O `pywinauto` com `backend="uia"` usa a **UI Automation**, a API de acessibilidade do Windows.

`Desktop(backend="uia")` representa a área de trabalho inteira. Dela pedimos uma janela por um
**padrão de título** (expressão regular), e `wait("visible")` substitui o `time.sleep` — é a mesma
ideia do `WebDriverWait` do Selenium.

In [ ]:
import re

if NO_WINDOWS:
    from pywinauto import Desktop


def escapar(texto: str) -> str:
    """Põe entre chaves os caracteres que o type_keys trata como comandos: + ^ % ~ { } ( )"""
    return re.sub(r"([{}()+^%~])", r"{\1}", texto)


def area_de_texto(janela):
    """A área de texto do Bloco de Notas: 'Document' no Windows 11, 'Edit' no Windows 10."""
    for tipo in ("Document", "Edit"):
        encontrados = janela.descendants(control_type=tipo)
        if encontrados:
            return encontrados[0]
    raise LookupError("Não achei a área de texto")


print("Ajudantes do Windows prontos." if NO_WINDOWS else "⏭️  (no Mac, esta célula só define funções que não serão usadas)")

In [ ]:
if NO_WINDOWS:
    nota_windows = SAIDA / "nota_pywinauto.txt"
    nota_windows.write_text("", encoding="utf-8")
    subprocess.Popen(["notepad.exe", str(nota_windows)])

    # O título é "nota_pywinauto.txt - Bloco de Notas" (ou "- Notepad" em inglês).
    bloco = Desktop(backend="uia").window(title_re=f".*{re.escape(nota_windows.name)}.*")
    bloco.wait("visible", timeout=15)
    print("Achei a janela:", bloco.window_text())
else:
    print("⏭️  Esta célula é só para Windows. No Mac, veja a seção 🍎 logo abaixo.")

#### Explorando a árvore

`print_control_identifiers()` lista tudo o que existe dentro da janela: tipo de controle
(`Button`, `Edit`, `Document`, `MenuItem`...), nome e `auto_id`. É o "Inspecionar elemento" do desktop.

> 💡 Para explorar com o mouse, o Windows SDK traz o **Inspect.exe**, e há também o
> [Accessibility Insights](https://accessibilityinsights.io/) (grátis): passe o mouse sobre
> qualquer coisa e veja tipo, nome e `auto_id`.

In [ ]:
if NO_WINDOWS:
    bloco.print_control_identifiers(depth=3)

#### Digitando e salvando

A área de texto é um controle do tipo `Document` no Bloco de Notas do Windows 11 e `Edit` no do
Windows 10. A função `area_de_texto()` tenta os dois.

Diferente do `pyautogui`, o `type_keys` do `pywinauto` **digita acentos sem problema**. Atenção só a
estes caracteres, que têm significado especial: `+` (Shift), `^` (Ctrl), `%` (Alt), `~` (Enter),
`{ } ( )`. Para digitá-los literalmente, ponha entre chaves: `{+}`. A função `escapar()` faz isso.

In [ ]:
if NO_WINDOWS:
    texto = "Olá! Este texto foi digitado pelo pywinauto.\nAcentos funcionam: ação, coração, pão de queijo.\n"
    editor = area_de_texto(bloco)
    editor.set_focus()
    editor.type_keys(escapar(texto), with_spaces=True, with_newlines=True, pause=0.01)

    bloco.type_keys("^s")  # Ctrl+S. O arquivo já existe, então salva direto, sem perguntar o nome.
    time.sleep(1)
    print("💾 Conteúdo salvo:\n" + nota_windows.read_text(encoding="utf-8"))
    bloco.close()

#### A Calculadora: botões pelo `auto_id`

Cada botão da Calculadora do Windows tem um **`auto_id`** fixo (`num7Button`, `plusButton`...) que
**não muda com o idioma** do Windows. O nome do botão mudaria ("Sete" × "Seven"); o `auto_id`, não.
É o equivalente desktop do `data-testid` que usamos no Selenium.

In [ ]:
if NO_WINDOWS:
    subprocess.Popen(["calc.exe"])
    calc = Desktop(backend="uia").window(title_re="Calculadora|Calculator")
    calc.wait("visible", timeout=15)

    for botao in ["clearButton", "num7Button", "plusButton", "num8Button", "multiplyButton", "num2Button", "equalButton"]:
        calc.child_window(auto_id=botao).click_input()

    visor = calc.child_window(auto_id="CalculatorResults").window_text()
    print("🧮", visor)  # "A exibição é 30" (a Calculadora padrão calcula na ordem: (7+8)×2)
    calc.close()
else:
    print("⏭️  Só para Windows.")

### 🍎 Mac: AppleScript

No Mac, muitos apps são **programáveis por AppleScript**: eles publicam um "dicionário" de objetos
(documentos, janelas, textos) e comandos. Do Python, chamamos o AppleScript pelo comando `osascript`.

> 💡 Para ver o dicionário de um app: abra o **Editor de Scripts** (vem no Mac) →
> Arquivo → Abrir Dicionário → escolha o app.

`with timeout` faz o comando desistir em vez de travar se o app estiver esperando alguma resposta sua.

In [ ]:
def applescript(script: str) -> str:
    """Roda um AppleScript e devolve o resultado como texto (só no Mac)."""
    resultado = subprocess.run(
        ["osascript", "-e", f"with timeout of 20 seconds\n{script}\nend timeout"],
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    if resultado.returncode != 0:
        raise RuntimeError(resultado.stderr.strip())
    return resultado.stdout.strip()


if NO_MAC:
    print(applescript('return "Olá do AppleScript! Hoje é " & (current date)'))
else:
    print("⏭️  Só para Mac.")

#### Escrevendo e salvando no TextEdit

O mesmo fluxo do Windows: o Python cria o arquivo, o sistema o abre no editor e o script
**encontra o documento pelo caminho** (não pela posição na tela), escreve o texto e salva.

> 🔒 Por que não mandar o TextEdit "salvar como" num caminho qualquer? O TextEdit roda numa
> *sandbox* e só pode gravar onde **você** deixou — abrindo o arquivo com `open`, damos essa
> permissão. Mandar salvar em outro lugar faz o comando travar esperando.

In [ ]:
if NO_MAC:
    nota_mac = SAIDA / "nota_applescript.txt"
    nota_mac.write_text("", encoding="utf-8")
    subprocess.run(["open", "-a", "TextEdit", str(nota_mac)], check=True)
    time.sleep(1)

    texto = "Olá! Este texto foi escrito via AppleScript.\nAcentos funcionam: ação, coração, pão de queijo."
    texto_as = texto.replace("\\", "\\\\").replace('"', '\\"')  # aspas viram \" dentro do AppleScript

    applescript(f"""
    tell application "TextEdit"
        set doc to first document whose path is "{nota_mac}"
        set text of doc to "{texto_as}"
        save doc
        close doc
    end tell
    """)
    print("💾 Conteúdo salvo:\n" + nota_mac.read_text(encoding="utf-8"))
else:
    print("⏭️  Só para Mac. No Windows, veja a seção 🪟 acima.")

#### Explorando a árvore no Mac: System Events

Para apps **sem** dicionário AppleScript, o caminho é o **System Events**, que enxerga a árvore de
acessibilidade de qualquer app — o equivalente ao `print_control_identifiers()`.

Ele exige a permissão de **Acessibilidade**. Se faltar, a célula explica o que fazer.

In [ ]:
if NO_MAC:
    try:
        menus = applescript(
            'tell application "System Events" to get name of every menu bar item of menu bar 1 '
            'of (first application process whose frontmost is true)'
        )
        print("Menus do app em primeiro plano:", menus)

        janelas = applescript(
            'tell application "System Events" to get name of every window of '
            '(every application process whose visible is true)'
        )
        print("Janelas abertas por app:", janelas)
    except RuntimeError as erro:
        if "-1719" in str(erro) or "assistive" in str(erro):
            print("🔒 Falta a permissão de Acessibilidade para o programa que roda este notebook.")
            print("   Ajustes do Sistema → Privacidade e Segurança → Acessibilidade → ative-o e reinicie-o.")
        else:
            raise
else:
    print("⏭️  Só para Mac.")

> 💡 Com System Events também se clica: `click menu item "Copiar" of menu "Editar" of menu bar 1`.
> Mas repare: os **nomes mudam com o idioma do sistema**. Em scripts para vários computadores,
> prefira o dicionário do app (como fizemos no TextEdit), que não depende do idioma.

---
## B.3 Juntando tudo: um relatório automático

Situação real: todo dia chega um CSV de vendas e alguém precisa montar um resumo e salvá-lo.
Vamos separar o programa em duas partes:
1. **a regra de negócio** (ler o CSV e montar o texto) — é Python puro, igual em qualquer sistema;
2. **o "motor" de interface** (escrever no editor) — escolhido conforme o sistema.

![O robô do relatório: CSV, regra de negócio, motor por sistema, arquivo conferido](diagramas/robo_relatorio.svg)

In [ ]:
import csv
from collections import defaultdict


def montar_relatorio(arquivo_csv: Path) -> str:
    por_vendedor = defaultdict(float)
    por_produto = defaultdict(int)
    with open(arquivo_csv, encoding="utf-8", newline="") as f:
        for linha in csv.DictReader(f):
            qtd = int(linha["quantidade"])
            por_vendedor[linha["vendedor"]] += qtd * float(linha["preco_unitario"])
            por_produto[linha["produto"]] += qtd

    linhas = ["RELATÓRIO DE VENDAS", "=" * 30, "", "Faturamento por vendedor:"]
    for vendedor, total in sorted(por_vendedor.items(), key=lambda kv: -kv[1]):
        linhas.append(f"  {vendedor:<10} R$ {total:>8.2f}")
    linhas += ["", "Unidades por produto:"]
    for produto, qtd in sorted(por_produto.items(), key=lambda kv: -kv[1]):
        linhas.append(f"  {produto:<15} {qtd:>3}")
    linhas += ["", f"Total geral: R$ {sum(por_vendedor.values()):.2f}"]
    return "\n".join(linhas)


relatorio = montar_relatorio(AQUI / "dados" / "vendas.csv")
print(relatorio)

In [ ]:
def escrever_no_editor(texto: str, arquivo: Path):
    """Abre o arquivo no editor do sistema, escreve o texto pela interface e salva."""
    arquivo.write_text("", encoding="utf-8")

    if NO_WINDOWS:
        subprocess.Popen(["notepad.exe", str(arquivo)])
        janela = Desktop(backend="uia").window(title_re=f".*{re.escape(arquivo.name)}.*")
        janela.wait("visible", timeout=15)
        editor = area_de_texto(janela)
        editor.set_focus()
        editor.type_keys(escapar(texto), with_spaces=True, with_newlines=True, pause=0.005)
        janela.type_keys("^s")
        time.sleep(1)
        janela.close()

    elif NO_MAC:
        subprocess.run(["open", "-a", "TextEdit", str(arquivo)], check=True)
        time.sleep(1)
        texto_as = texto.replace("\\", "\\\\").replace('"', '\\"')
        applescript(f"""
        tell application "TextEdit"
            set doc to first document whose path is "{arquivo}"
            set text of doc to "{texto_as}"
            save doc
            close doc
        end tell
        """)

    else:
        raise NotImplementedError(f"Sem motor de interface para {SISTEMA}")


destino = SAIDA / "relatorio_vendas.txt"
escrever_no_editor(relatorio, destino)
conferido = destino.read_text(encoding="utf-8").replace("\r\n", "\n").strip()
assert conferido == relatorio.strip(), "O arquivo salvo não bate com o relatório!"
print(f"✅ Relatório escrito pela interface e conferido: {destino}")

Repare na última linha: **conferimos o resultado** com `assert`, como nos testes da Parte A.
Automação sem verificação é só torcer para dar certo.

## 🎯 Exercício B1

1. Acrescente ao relatório uma seção **"Vendas por dia"**.
2. Use o `pyautogui` para tirar uma captura de tela **com o editor aberto** antes de fechá-lo e salve
   em `saida/`.
3. (Windows) Na Calculadora, calcule `123 × 4` pelos `auto_id`s.
   (Mac) Use o System Events para listar os **itens do menu Arquivo** do TextEdit.

<details>
<summary>👀 Dicas</summary>

```python
# 1. dentro de montar_relatorio, conte também:  por_dia[linha["data"]] += qtd * preco
# 3. Mac:
applescript('tell application "System Events" to get name of every menu item of menu 3 '
            'of menu bar 1 of application process "TextEdit"')
```
</details>

In [ ]:
# Sua resposta aqui

---
# Resumo

| Preciso automatizar... | Use | Ache os elementos por | Espere com |
|---|---|---|---|
| um site | **Selenium** | `By.ID`, `data-testid`, CSS | `WebDriverWait` |
| um app do Windows | **pywinauto** (`uia`) | `auto_id`, `control_type` | `.wait("visible")` |
| um app do Mac | **AppleScript** (`osascript`) | dicionário do app; System Events | `with timeout` |
| qualquer coisa na tela | **pyautogui** | coordenadas, imagens | `time.sleep` 😬 |

Três lições que valem para todas:
1. **Ache as coisas pelo que elas são**, não por onde estão.
2. **Espere por condições**, não por tempo fixo.
3. **Verifique o resultado** — um teste que não tem `assert` não testa nada.

## Para continuar estudando

- [Documentação do Selenium](https://www.selenium.dev/documentation/) · [Selenium com Python](https://selenium-python.readthedocs.io/)
- [pytest](https://docs.pytest.org/) · [Playwright para Python](https://playwright.dev/python/) (alternativa moderna ao Selenium)
- [pyautogui](https://pyautogui.readthedocs.io/) · [pywinauto](https://pywinauto.readthedocs.io/)
- [Guia de AppleScript da Apple](https://developer.apple.com/library/archive/documentation/AppleScript/Conceptual/AppleScriptLangGuide/)